##**[4주차]과제**
- 아래의 과제1), 과제2)의 코드를 완성하시오.
- 모든 코드의 결과를 출력하여, .ipynb의 링크를 **[4주차]/[4주차]과제**에 제출하시오.\
(실습 제출 예시: 4주차_2020XXXX_이름.ipynb 코드 링크)

In [99]:
# 학번, 이름 출력
print(f'학번 : 2343035 이름 : 신동엽')

학번 : 2343035 이름 : 신동엽


In [100]:
# google drive 연결
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [101]:
# 구글 드라이브에 데이터를 넣고 path 연결 필요
data_path = '/content/drive/MyDrive/deepLearning'

## 과제 1
- 케라스를 이용하여 은닉층이 2개인 MLP를 생성하고 옵티마이저를 Adam, RMSprop, Adadelta, Adagrad로 변경해보면서 성능을 측정한 뒤 어떤 방법이 가장 좋은 성능을 보이는지 확인하고, 각 방법에 대해서 서술하시오.
- 은닉층의 노드 수는 512, 256로 설정
- 은닉층의 Activation Function은 ReLU로 설정
- 모델 레이어 추가  
model.add(tf.keras.layers.Dense(노드갯수, activation='activation_func', kernel_initializer=initializer))  
<출력 예시>  
테스트 정확도 : 0.2779000282287598


In [102]:
# 런타임 재시작 필요
!pip install scikeras==0.13.0 scikit-learn==1.4.2

In [103]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import GridSearchCV
from scikeras.wrappers import KerasClassifier, KerasRegressor
import os
import numpy as np

In [104]:
# 실험 결과 재현을 위한 seed 고정
SEED = 4
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
tf.random.set_seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
initializer = tf.keras.initializers.GlorotUniform(seed=SEED)

In [105]:
# 데이터세트 준비
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

In [106]:
# 데이터 전처리
train_images = train_images.reshape((60000, 784))
train_images = train_images.astype('float32') / 255

test_images = test_images.reshape((10000, 784))
test_images = test_images.astype('float32') / 255

train_labels = tf.keras.utils.to_categorical(train_labels)
test_labels = tf.keras.utils.to_categorical(test_labels)

In [107]:
# 신경망 모델 구축
def build_model():
    network = tf.keras.models.Sequential()

    # 은닉층 생성
    # 여기에 코드를 작성해주세요
    network.add(tf.keras.layers.Dense(512, activation='relu', kernel_initializer=initializer)) #노드 512 은닉층 추가
    network.add(tf.keras.layers.Dense(256, activation='relu', kernel_initializer=initializer)) #노드 256 은닉층 추가
    network.add(tf.keras.layers.Dense(10, activation='sigmoid', kernel_initializer=initializer))

    # 옵티마이저 설정 필요(Adam, RMSprop, Adadelta, Adagrad)
    network.compile(optimizer='Adagrad',
                   loss='mse',
                   metrics=['accuracy'])
    return network

In [108]:
# model 생성
model = build_model()

In [109]:
result = model.fit(train_images, train_labels, epochs=5, batch_size=64)

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.0758 - loss: 0.1619
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.1534 - loss: 0.0987
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.3022 - loss: 0.0912
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.3878 - loss: 0.0889
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.4217 - loss: 0.0874


In [110]:
test_loss, test_acc = model.evaluate(test_images, test_labels)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.4393 - loss: 0.0864


In [111]:
print('테스트 정확도 : ', test_acc)

테스트 정확도 :  0.439300000667572


### 각 옵티마이저의 성능 작성
- RMSprop : 0.9761000275611877
- Adagrad : 0.439300000667572
- Adadelta : 0.37700000405311584
- Adam : 0.9818999767303467

### 가장 좋은 성능의 옵티마이저 작성
- Adam

### 옵티마이저 서술
- RMSprop :
adagrad의 단점을 보완한 방법, gradient 누적 대신 EMA(exponential moving average; 지수 가중 이동 평균)를 사용하여 학습률 조정
최근 데이터에는 더 큰 가중치를 주고, 과거 데이터에는 지수적으로 감소하는 가중치를 주는 평균 이동 방법, 학습률이 너무 빨리 감소하는 문제 해결

- Adagrad :  학습률을 각 매개변수에 대해 적응적으로 조정하는 방법
자주 학습된 방향은 덜 움직이고, 덜 학습된 방향은 더 많이 움직임 -> 학습률 감쇠(learning rate decay)
학습률 설정: 이전단계의 기울기들을 누적한 값에 반비례하여 설정

- Adadelta :
Adagrad의 학습률 소실 문제를 해결하기 위해 제안된 방법. 기울기 제곱을 무한히 누적하는 대신, 지수 이동 평균을 사용하여 과거의 기울기 정보가 미치는 영향을 점진적으로 감소시킴

- Adam :
momentum과 RMSprop을 결합한 방법, 적응적 학습률과 모멘텀을 모두 활용하여 가중치 업데이트, 현대 딥러닝에서 가장 널리 사용되는 optimizer

## 과제 2 MLP 모델의 정확도(Accuracy)를 71이상 달성하시오.  
- 해당 diabetes.csv 데이터는 Pima 인디언의 의료 기록과 각 환자가 5년 이내에 당뇨병 발병 여부가 저장되어 있다.
- 현재 모델은 1 : 당뇨병 양성, 0 : 당뇨병 음성을 분류하는 MLP 모델이다.
- 해당 모델의 하이퍼 파라미터를 수정하여 정확도 71 이상을 달성하여라.
- 하이퍼 파라미터 : Activation Function, optimizer, epochs, batch size 등


In [112]:
!pip install pandas

In [113]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adagrad, Adadelta, RMSprop, Adam
import pandas as pd
import numpy as np
import os

In [114]:
# 데이터 load
dataset = pd.read_csv(os.path.join(data_path, 'diabetes.csv'))

X = dataset.iloc[:, 0:8].values
y = dataset.iloc[:, 8].values

In [115]:
# 수정 x
def configure_tf_determinism():
    # TensorFlow의 병렬성 감소
    tf.config.experimental.enable_op_determinism()

In [116]:
# 실험 결과 재현을 위한 seed 고정
SEED = 4
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
tf.random.set_seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
initializer = tf.keras.initializers.GlorotUniform(seed=SEED)

In [117]:
# 모델 생성
def build_model():
    model = Sequential()
    # 수정 가능
    model.add(tf.keras.layers.Dense(12, input_dim=8, activation='relu', kernel_initializer=initializer))
    model.add(tf.keras.layers.Dense(8, activation='relu', kernel_initializer=initializer))

    # 수정 x
    model.add(tf.keras.layers.Dense(1, activation='sigmoid', kernel_initializer=initializer))

    model.compile(loss='binary_crossentropy', optimizer='Adam', metrics=['accuracy'])

    return model

In [118]:
# 수정 x
configure_tf_determinism()

In [119]:
model = build_model()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [120]:
model.fit(X, y, epochs=5, batch_size=1)

Epoch 1/5
768/768 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5768 - loss: 1.0543
Epoch 2/5
768/768 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.6432 - loss: 0.7325
Epoch 3/5
768/768 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.6745 - loss: 0.6736
Epoch 4/5
768/768 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6810 - loss: 0.6419
Epoch 5/5
768/768 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6797 - loss: 0.6282


In [121]:
_, accuracy = model.evaluate(X, y)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7240 - loss: 0.5740  


In [122]:
print('정확도: %.2f' % (accuracy*100))

정확도: 72.40
